# 04 - GNN training: ligand GIN + ESM pocket-pool + MLP

Same data split as notebook 03 (PDBbind v2020 refined minus CASF-2016, 90/10 train/val). The ligand encoder is now a learned GINE graph network instead of fixed ECFP fingerprints.

**Architecture**: ligand graph -> GINE encoder -> mean+max pool, then concat with ESM-2 pocket/whole embeddings -> MLP -> scalar pK.

Training uses MSE loss, AdamW, `ReduceLROnPlateau` on val Pearson R, early stop after 15 epochs.

In [1]:
from __future__ import annotations

import json
import sys
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
REPORTS_ROOT = PROJECT_ROOT / "reports"
RUNS_ROOT = PROJECT_ROOT / "runs" / "gnn"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from plb.data.dataset import PDBbindGraphDataset, load_ligand_graph_cache, make_dataloader
from plb.data.pdbbind import find_default_paths
from plb.eval import format_metrics_row, regression_metrics
from plb.models.gnn import AffinityModel, GNNConfig
from plb.train import TrainConfig, evaluate, train

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} on {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch 2.6.0+cu124 on cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## 1. Build the datasets

Re-using the split from `plb data prepare` and the precomputed ligand graph + ESM caches.

In [2]:
paths = find_default_paths(DATA_ROOT)
refined_root = paths["refined_root"]

idx = pd.read_csv(DATA_ROOT / "processed" / "refined_index.csv")
labels = dict(zip(idx["pdb_id"].str.lower(), idx["pK"], strict=True))

split_blob = json.loads((DATA_ROOT / "processed" / "split.json").read_text(encoding="utf-8"))
train_ids = [p.lower() for p in split_blob["train"]]
val_ids = [p.lower() for p in split_blob["val"]]
test_ids = [p.lower() for p in split_blob["test"]]
print(f"split sizes: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}")

ligand_cache = load_ligand_graph_cache(DATA_ROOT)
print(f"ligand graph cache: {len(ligand_cache)} entries")

train_ds = PDBbindGraphDataset(
    train_ids, labels, DATA_ROOT, refined_root, ligand_cache=ligand_cache
)
val_ds = PDBbindGraphDataset(val_ids, labels, DATA_ROOT, refined_root, ligand_cache=ligand_cache)
test_ds = PDBbindGraphDataset(test_ids, labels, DATA_ROOT, refined_root, ligand_cache=ligand_cache)
print(f"materialised: train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

skipped = train_ds.skipped + val_ds.skipped + test_ds.skipped
if skipped:
    reasons = pd.Series([r for _, r in skipped]).value_counts()
    print("skipped:")
    print(reasons)

print("\nfirst train sample:")
print(train_ds[0])

split sizes: train=4545  val=505  test=266
ligand graph cache: 5316 entries
materialised: train=4545  val=505  test=266

first train sample:
Data(x=[10, 36], edge_index=[2, 20], edge_attr=[20, 8], y=[1], esm_pocket=[1, 480], esm_whole=[1, 480], pdb_id='184l')


## 2. Train

| Config  | hidden | layers | dropout |
|---------|--------|--------|---------|
| `gnn_a` | 128    | 3      | 0.10    |


In [3]:
BASE_TRAIN = TrainConfig(
    batch_size=64,
    max_epochs=80,
    learning_rate=1e-3,
    weight_decay=1e-4,
    early_stop_patience=15,
    scheduler_patience=5,
    seed=42,
    device=device,
    num_workers=0,
)

configs: dict[str, TrainConfig] = {}

cfg = deepcopy(BASE_TRAIN)
cfg.name = "gnn_a"
cfg.model = GNNConfig(hidden_dim=128, n_gnn_layers=3, dropout=0.10)
configs["gnn_a"] = cfg

for name, c in configs.items():
    n = AffinityModel(c.model).n_parameters()
    print(
        f"{name}: hidden={c.model.hidden_dim:3d}  layers={c.model.n_gnn_layers}  dropout={c.model.dropout:.2f}  params={n:,}"
    )

gnn_a: hidden=128  layers=3  dropout=0.10  params=451,332


In [4]:
summaries: dict[str, dict] = {}
for name, cfg in configs.items():
    print(f"\n=== {name} ===")
    summaries[name] = train(cfg, train_ds, val_ds, test_ds=test_ds, run_dir=RUNS_ROOT)

rows = []
for name, summary in summaries.items():
    test_metrics = summary.get("test_metrics", {})
    rows.append(
        {
            "config": name,
            "best_epoch": summary["best_epoch"],
            "val_pearson_r": summary["best_val_pearson_r"],
            "test_pearson_r": test_metrics.get("pearson_r"),
            "test_spearman_r": test_metrics.get("spearman_r"),
            "test_rmse": test_metrics.get("rmse"),
            "test_mae": test_metrics.get("mae"),
        }
    )
summary_df = pd.DataFrame(rows).set_index("config")
summary_df.round(4)


=== gnn_a ===


c:\Users\fearg\Protein_Ligand_Binding\.venv\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


,best_epoch,val_pearson_r,test_pearson_r,test_spearman_r,test_rmse,test_mae
config,,,,,,
gnn_a,25,0.7316,0.7423,0.7206,1.7249,1.38


## 3. Training curves

In [5]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("loss", "validation correlation"))
for name in configs:
    log_path = RUNS_ROOT / name / "log.jsonl"
    if not log_path.is_file():
        continue
    rows = [
        json.loads(line)
        for line in log_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not rows:
        continue
    df = pd.DataFrame(rows)
    fig.add_trace(
        go.Scatter(x=df["epoch"], y=df["train_loss"], mode="lines",
                   line=dict(dash="dash"), opacity=0.5, name=f"{name} train"),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=df["epoch"], y=df["val_loss"], mode="lines", name=f"{name} val"),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=df["epoch"], y=df["val_pearson_r"], mode="lines", name=name),
        row=1, col=2,
    )

fig.add_hline(y=0.0, line_width=1, line_color="black", row=1, col=2)
fig.update_xaxes(title_text="epoch")
fig.update_yaxes(title_text="MSE (pK units)", row=1, col=1)
fig.update_yaxes(title_text="val Pearson R", row=1, col=2)
fig.update_layout(height=400, width=1100, template="plotly_white")
fig.show()

## 4. Evaluate on CASF-2016

In [10]:
best_name = max(summaries, key=lambda k: summaries[k]["best_val_pearson_r"])
best_summary = summaries[best_name]
best_cfg = configs[best_name]
print(f"best config: {best_name}")
print(f"  val Pearson R = {best_summary['best_val_pearson_r']:.4f}")

model = AffinityModel(best_cfg.model).to(device)
ckpt = torch.load(best_summary["best_checkpoint"], map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_loader = make_dataloader(test_ds, batch_size=best_cfg.batch_size, shuffle=False)
test_eval = evaluate(model, test_loader, torch.device(device))
metrics = test_eval["metrics"]
print(f"CASF-2016: {format_metrics_row(metrics)}  (n={len(test_ds)})")

baseline_raw = json.loads((PROJECT_ROOT / "runs" / "baseline" / "metrics.json").read_text())
# baseline metrics.json uses capital-R keys (pearson_R, spearman_R)
key_map = {"pearson_r": "pearson_R", "spearman_r": "spearman_R", "rmse": "rmse", "mae": "mae"}
metrics_rows = {
    "baseline": {k: baseline_raw[key_map[k]] for k in ("pearson_r", "spearman_r", "rmse", "mae")},
    "phase4": {k: float(metrics[k]) for k in ("pearson_r", "spearman_r", "rmse", "mae")},
}
comparison = pd.DataFrame(metrics_rows).T
comparison["delta_pearson"] = comparison["pearson_r"] - comparison.loc["baseline", "pearson_r"]
comparison.round(4)

best config: gnn_a
  val Pearson R = 0.7316
CASF-2016: Pearson R = 0.742 | Spearman R = 0.721 | RMSE = 1.725 | MAE = 1.380  (n=266)


,pearson_r,spearman_r,rmse,mae,delta_pearson
baseline,0.7189,0.6865,1.5144,1.1837,0.0000
phase4,0.7423,0.7206,1.7249,1.3800,0.0234


In [8]:
y_true = test_eval["y_true"]
y_pred = test_eval["y_pred"]

lo = float(min(y_true.min(), y_pred.min())) - 0.5
hi = float(max(y_true.max(), y_pred.max())) + 0.5

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=y_true, y=y_pred, mode="markers",
        marker=dict(size=6, opacity=0.5),
        name="predictions",
    )
)
fig.add_trace(
    go.Scatter(
        x=[lo, hi], y=[lo, hi], mode="lines",
        line=dict(color="black", width=1, dash="dash"),
        name="y = x",
    )
)
fig.update_layout(
    xaxis=dict(title="measured pK", range=[lo, hi]),
    yaxis=dict(title="predicted pK", range=[lo, hi], scaleanchor="x"),
    title=f"CASF-2016: {best_name}  (Pearson R = {metrics['pearson_r']:.3f}, RMSE = {metrics['rmse']:.3f}, n = {len(y_true)})",
    width=550, height=550,
    template="plotly_white",
)

scatter_path = REPORTS_ROOT / "gnn_casf2016_scatter.png"
fig.write_image(str(scatter_path), scale=2)
fig.show()
print(f"saved: {scatter_path}")

saved: c:\Users\fearg\Protein_Ligand_Binding\reports\gnn_casf2016_scatter.png


In [9]:
summary_path = RUNS_ROOT / "sweep_summary.json"
summary_path.write_text(
    json.dumps(
        {
            "best_config": best_name,
            "best_val_pearson_r": float(best_summary["best_val_pearson_r"]),
            "best_test_metrics": {k: float(v) for k, v in metrics.items()},
            "per_config": {
                name: {
                    "best_epoch": int(s["best_epoch"]),
                    "val_pearson_r": float(s["best_val_pearson_r"]),
                    **(
                        {f"test_{k}": float(v) for k, v in s.get("test_metrics", {}).items()}
                        if s.get("test_metrics") is not None
                        else {}
                    ),
                }
                for name, s in summaries.items()
            },
            "n_train": len(train_ds),
            "n_val": len(val_ds),
            "n_test": len(test_ds),
            "device": str(device),
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(f"saved sweep summary: {summary_path}")

saved sweep summary: c:\Users\fearg\Protein_Ligand_Binding\runs\gnn\sweep_summary.json
